# Development notebook to unpack ambiguous toponym outputs

In [ ]:
from typing import Final
from pathlib import Path
from os import getenv
from dotenv import find_dotenv, load_dotenv
import pickle
import re
import json
import numpy as np
from PIL import Image
import shapely as sh
import shapely.plotting as shp
import pandas as pd
import geopandas as gp
import matplotlib.pyplot as plt
from outputs import convert_ToponymExtractor_outputs_to_gdf
from edina import get_transformer_from_geodataframe

PROJECT_DIR: Final[Path] = Path(find_dotenv(".env", 1, 1)).absolute().parent
load_dotenv(PROJECT_DIR.joinpath(".env"))
LOCAL_DIR: Final[Path] = Path(getenv("LOCAL_DIR"))
IMG_DIR = LOCAL_DIR\
    .joinpath("outputs/toponym-extractor-post-processing/Ambiguous")

# Loading ToponymExtractor outputs
outputs = []
ambiguous_fp = LOCAL_DIR\
    .joinpath("outputs/toponym-extractor-ambiguous-masks/ambiguous.pkl")
with open(ambiguous_fp, "rb") as f:
    try:
        while 1:
            outputs.append(pickle.load(f))
    except EOFError as _:
        # all outputs have been read, context will automatically close
        pass
    except Exception as e:
        raise

In [ ]:
outputs, errors = convert_ToponymExtractor_outputs_to_gdf(outputs, 2000, 2000)

In [ ]:
getattr(errors, "png_filename", [])

In [ ]:
outputs

## Inspecting ToponymExtractor outputs

In [ ]:
filepattern = re.compile(r"[-\w]+-ambiguous")

In [ ]:
image_filenames = outputs.png_filename.unique().tolist()
example_idx = 1
example = image_filenames[example_idx]
example

In [ ]:
# get image and metadata
img = np.array(Image.open(IMG_DIR.joinpath(example)))
metadata = filepattern.search(example).group() + "-meta.json"
with open(IMG_DIR.joinpath(metadata), "r") as f:
    metadata = json.load(f)

In [ ]:
# get snippet boxes for ambiguous image
img_meta = metadata[example]
img_meta

In [ ]:
# constrain outputs to example case
outputs_eg = outputs.loc[outputs.png_filename == example].copy()
outputs_eg

In [ ]:
boxes = []
box_iter = zip(
    img_meta["cliques"],
    img_meta["row_pos"],
    img_meta["col_pos"],
    img_meta["shapes"],
    strict = True
)
for clique, row, col, (h, w) in box_iter:
    boxes.append({
        "clique_id": clique,
        "minx": col,
        "miny": row,
        "maxx": col + w - 1,
        "maxy": row + h - 1,
        "geometry": sh.box(col, row, col + w - 1, row + h - 1)
    })
boxes = gp.GeoDataFrame(boxes)
boxes

In [ ]:
# inspect image and predictions data and bouding boxes
fig, ax = plt.subplots(figsize = (15, 15))
ax.imshow(img, cmap = "grey")
boxes.plot(ax = ax, color = "blue", alpha = .5)
outputs_eg.plot(ax = ax, color = "red", alpha = .5)

Ouputs appear to have a positional bias, which is interesting.

In [ ]:
output_clique = gp.sjoin(
    outputs.loc[outputs.png_filename == example].copy(),
    boxes[["clique_id", "geometry"]],
    how = "inner",
    predicate = "intersects"
)
output_clique.sort_values("clique_id")

In [ ]:
fig, ax = plt.subplots(figsize = (15, 15))
ax.imshow(img, cmap = "grey")
boxes.plot(ax = ax, color = "blue", alpha = .5)
output_clique.plot(ax = ax, color = "red", alpha = .5)


In [ ]:
img_preds = output_clique.copy()
# clip image predictions geometries to the bounds of
# the PNG snippet
selection = boxes.loc[img_preds.index_right.to_list(), "geometry"]
img_preds["geometry"] = img_preds\
    .geometry.intersection(selection, align = False)
# remove offset on prediction coordinates caused by concatenating images
for tup in boxes.itertuples(index = False):
    selection = (img_preds.clique_id == tup.clique_id)
    img_preds.loc[selection, "geometry"] = img_preds\
        .loc[selection, "geometry"]\
        .transform(lambda x: x - [tup.minx, tup.miny])

In [ ]:
fig, ax = plt.subplots(figsize = (15, 15))
ax.imshow(img, cmap = "grey")
boxes.plot(ax = ax, color = "blue", alpha = .5)
img_preds.plot(ax = ax, color = "red", alpha = .5)


In [ ]:
img_preds.bounds

## Inspect control points

In [ ]:
gcp = gp.read_file(IMG_DIR.joinpath("control-points.gpkg"))
gcp

In [ ]:
# Create georeferenced polygons
cliques =  img_preds.clique_id.unique()
clique_eg = cliques[0]

gcp_trans = gcp.loc[(
    (gcp.tiff_name == (img_meta["tiff_stem"] + ".tif"))
    & (gcp.clique_idx == clique)
)]
print(gcp_trans.head())
gcp_trans = get_transformer_from_geodataframe(gcp_trans)

In [ ]:
# Get example polygon
geo_eg: gp.GeoSeries = img_preds\
    .loc[img_preds.clique_id == clique_eg]\
    .geometry

poly_eg: sh.Polygon = geo_eg.iloc[0]
poly_eg

In [ ]:
coords_eg = sh.get_coordinates(poly_eg)
gcp_trans.xy(coords_eg[:, 1], coords_eg[:, 0])

In [ ]:
geo_eg.transform(lambda x:  np.array([*zip(*gcp_trans.xy(x[:, 1], x[:, 0]))]))

In [ ]:
# Documentation recommends calling close on
# GCPTransformer after calling transforms
gcp_trans.close()

In [ ]:
# Create georeferenced polygons
img_preds_geo = img_preds.copy()
for clique in img_preds_geo.clique_id.unique():
    gcp_trans = gcp.loc[(
        (gcp.tiff_name == (img_meta["tiff_stem"] + ".tif"))
        & (gcp.clique_idx == clique)
    )]
    gcp_trans =\
        get_transformer_from_geodataframe(gcp_trans)
    
    # Convert pixel location coordinates to latitude/
    # longitude
    selection = (img_preds_geo.clique_id == clique)
    img_preds_geo.loc[selection, "geometry"] = img_preds_geo\
        .loc[selection, "geometry"]\
        .transform(lambda x:  np.array([*zip(*gcp_trans.xy(x[:,1], x[:,0]))]))
    # Documentation recommends calling close on
    # GCPTransformer after calling transforms
    gcp_trans.close()

img_preds_geo = img_preds_geo.set_crs(gcp.crs)
img_preds_geo.head()

## Compare with original cliques.

In [ ]:
original_preds = gp\
    .read_file(IMG_DIR.joinpath(img_meta["tiff_stem"] + "-cliques.gpkg"))\
    .rename(columns = {"clique_idx": "clique_id"})
original_preds.head()

In [ ]:
original_centroids = original_preds.copy()
original_centroids["geometry"] = original_centroids.centroid

pairs = gp.sjoin(
    img_preds_geo.drop(columns = ["index_right"]),
    original_preds.drop(columns = ["png_filename", "key"]),
    how = "inner",
    predicate = "intersects",
    on_attribute = "clique_id"
)
pairs.index.unique().sort_values()